# Laboratorio 8. $k$-Nearest Neighbors (KNN)

## Carga y visualización de datos

Consideremos observaciones pertenecientes a dos clases. KNN determina la clase de una nueva observación a partir de las clases de sus vecinos más cercanos.

In [ ]:
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt

X, y = make_blobs(
    n_samples=100,
    centers=2,
    cluster_std=3,
    random_state=42
)

plt.scatter(X[:, 0], X[:, 1], c=y)

plt.xlabel("X1")
plt.ylabel("X2")
plt.show()

## Ajuste del modelo $k$-NN

Ajustamos el modelo utilizando $k=5$, por lo que la clasificación de una nueva observación dependerá de sus cinco vecinos más cercanos.

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X, y)

## Predicción de una nueva observación

Identificamos los $k$ vecinos más cercanos a la nueva observación y asignamos la clase con mayor representación entre ellos.

In [ ]:
new_point = np.array([[0, 4]])

prediction = knn.predict(new_point)

print("Clase predicha:", prediction[0])

In [ ]:
distances, indices = knn.kneighbors(new_point)

plt.scatter(X[:, 0], X[:, 1], c=y)

# Nueva observación
plt.scatter(new_point[:, 0], new_point[:, 1], marker="X",
    s=100, label="Nueva observación"
)

# Vecinos utilizados por KNN
plt.scatter(X[indices[0], 0], X[indices[0], 1], s=180,
    facecolors="none",
    edgecolors="black",
    linewidths=1.5,
    label="Vecinos"
)

plt.xlabel("X1")
plt.ylabel("X2")
plt.legend()
plt.show()

## Clasificación con un conjunto de datos real

Utilizaremos el conjunto de datos Wine, que contiene características químicas de vinos pertenecientes a tres clases diferentes.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

data = load_wine()

X = data.data
y = data.target

print(X.shape)
print(data.target_names)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Selección de $k$ mediante validación cruzada

Utilizamos validación cruzada para comparar diferentes valores de $k$ y seleccionar aquel con mejor desempeño promedio.

In [ ]:
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

k_values = range(1, 21)

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

mean_scores = []

for k in k_values:

    fold_scores = []

    for train_index, val_index in kf.split(X_train, y_train):

        X_fold_train = X_train[train_index]
        X_fold_val = X_train[val_index]

        y_fold_train = y_train[train_index]
        y_fold_val = y_train[val_index]

        # Estandarización
        scaler = StandardScaler()

        X_fold_train = scaler.fit_transform(X_fold_train)
        X_fold_val = scaler.transform(X_fold_val)

        # Modelo
        knn = KNeighborsClassifier(n_neighbors=k)

        knn.fit(X_fold_train, y_fold_train)

        # Predicción
        y_pred = knn.predict(X_fold_val)

        fold_scores.append(accuracy_score(y_fold_val, y_pred))

    mean_scores.append(np.mean(fold_scores))

## Comparación de $k$

In [ ]:
import matplotlib.pyplot as plt

plt.plot(k_values, mean_scores, marker="o")

plt.xlabel("Número de vecinos (k)")
plt.ylabel("Accuracy promedio")
plt.xticks(k_values)

plt.show()

## Selección de $k$

Seleccionamos el valor de $k$ con mayor desempeño promedio durante la validación cruzada.

In [ ]:
best_index = np.argmax(mean_scores)
best_k = list(k_values)[best_index]

print("Mejor k:", best_k)
print("CV accuracy:", mean_scores[best_index])

## Ajuste del modelo

Una vez seleccionado $k$, ajustamos el modelo utilizando todos los datos de entrenamiento.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

final_knn = KNeighborsClassifier(n_neighbors=best_k)

final_knn.fit(X_train_scaled, y_train)

## Evaluación del modelo

Finalmente, evaluamos el modelo sobre el conjunto de prueba, que no participó en la selección de $k$.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

y_pred = final_knn.predict(X_test_scaled)

print("Test accuracy:", accuracy_score(y_test, y_pred))

print("Matriz de confusión:\n", confusion_matrix(y_test, y_pred))